<a href="https://colab.research.google.com/github/laxmiboga9220-dev/Food_Image_Classification_and_Calorie_Estimation/blob/main/FOOD_RECOGNITION.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# 0. Install Dependencies
# ============================================================
# timm  -> for EfficientNet
# grad-cam -> for visualization
# scikit-learn -> for extra evaluation metrics
!pip install timm grad-cam scikit-learn

import os, shutil, urllib.request
import pandas as pd
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
from tqdm import tqdm
import random

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

import timm
from google.colab import files
from google.colab import drive

# Extra metrics from scikit-learn
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    accuracy_score,
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)

In [ ]:
# ============================================================
# 0.1 Mount Google Drive (for saving models)
# ============================================================
# This mounts your Google Drive to /content/drive
# You will be asked to authorize access when you run this cell.
drive.mount('/content/drive')

# Directory inside Google Drive where models will be saved.
# You can change this path if you want.
SAVE_DIR = "/content/drive/MyDrive/food101_models"
os.makedirs(SAVE_DIR, exist_ok=True)

BEST_MODEL_PATH = os.path.join(SAVE_DIR, "food_model_best.pth")
FINAL_MODEL_PATH = os.path.join(SAVE_DIR, "food_model_final.pth")

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)
print("Models will be saved to:", SAVE_DIR)



In [ ]:
# ============================================================
# 1. Download Food-101 Dataset
# ============================================================
url = "http://data.vision.ee.ethz.ch/cvl/food-101.tar.gz"
fname = "food-101.tar.gz"

# Download only if the file doesn't already exist
if not os.path.exists(fname):
    print("Downloading Food-101...")
    urllib.request.urlretrieve(url, fname)
else:
    print("food-101.tar.gz already exists - skipping download.")

print("Extracting dataset (this may take a while)...")
# Extract the tar.gz file (creates 'food-101' folder)
!tar -xzf food-101.tar.gz

base = "food-101"
img_path = f"{base}/images"


In [ ]:
# ============================================================
# 2. Select 10 Classes
# ============================================================
# We only use a subset of 10 classes to reduce training time.
SELECTED_CLASSES = [
    "pizza",
    "fried_rice",
    "chicken_curry",
    "sushi",
    "cup_cakes",
    "ice_cream",
    "pasta",
    "hamburger",
    "omelette",
    "grilled_salmon"
]

print("Using classes:", SELECTED_CLASSES)


In [ ]:
# ============================================================
# 3. Create folders dataset/train + dataset/val
# ============================================================
# We will copy images into a smaller custom folder structure:
# dataset/
#   train/
#       class_1/
#       ...
#   val/
#       class_1/
#       ...
for split in ["train", "val"]:
    for cls in SELECTED_CLASSES:
        os.makedirs(f"dataset/{split}/{cls}", exist_ok=True)


In [ ]:
# ============================================================
# 4. Load original train/test splits
# ============================================================
# The original Food-101 dataset provides text files listing
# which images belong to train and test.
def read_split(path):
    items = []
    with open(path) as f:
        for line in f:
            cls, img = line.strip().split("/")
            items.append((cls, img))
    return items

train_list = read_split(f"{base}/meta/train.txt")
val_list   = read_split(f"{base}/meta/test.txt")


In [ ]:
# ============================================================
# 5. Copy only selected classes
# ============================================================
# We copy only the images that belong to SELECTED_CLASSES.
def copy_subset(items, split):
    for cls, img in tqdm(items, desc=f"Copying {split} images"):
        if cls in SELECTED_CLASSES:
            src = f"{img_path}/{cls}/{img}.jpg"
            dst = f"dataset/{split}/{cls}/{img}.jpg"
            if os.path.exists(src):
                shutil.copy(src, dst)

copy_subset(train_list, "train")
copy_subset(val_list,   "val")

In [ ]:
# ============================================================
# 6. Create calorie CSV files
# ============================================================
# We assign a fixed calorie value per class (simple assumption).
calorie_map = {
    "pizza": 266,
    "fried_rice": 163,
    "chicken_curry": 240,
    "sushi": 200,
    "cup_cakes": 305,
    "ice_cream": 207,
    "pasta": 131,
    "hamburger": 295,
    "omelette": 154,
    "grilled_salmon": 208
}

def make_csv(split):
    rows = []
    root = f"dataset/{split}"
    # For each class, we list all image filenames and map to class + calories
    for cls in SELECTED_CLASSES:
        class_dir = os.path.join(root, cls)
        for fname in os.listdir(class_dir):
            rows.append([fname, cls, calorie_map[cls]])
    df = pd.DataFrame(rows, columns=["filename", "label", "calories"])
    df.to_csv(f"{split}_calories.csv", index=False)
    print(f"Saved {split}_calories.csv with {len(df)} rows.")

make_csv("train")
make_csv("val")



In [ ]:
# ============================================================
# 7. Dataset Class
# ============================================================
# We map string labels to integer indices and back.
label_to_idx = {c: i for i, c in enumerate(SELECTED_CLASSES)}
idx_to_label = {i: c for c, i in label_to_idx.items()}

class FoodDataset(Dataset):
    """
    Custom Dataset for loading images + label index + calories
    from our 'dataset/{split}' folders and '{split}_calories.csv' files.
    """
    def __init__(self, root, csv_file, tf=None):
        self.root = root
        self.df = pd.read_csv(csv_file)
        self.tf = tf

    def __len__(self):
        return len(self.df)

    def __getitem__(self, i):
        fname, label, cal = self.df.iloc[i]
        # Build full path: root/label/filename
        path = os.path.join(self.root, label, fname)
        # Load image and convert to RGB
        img = Image.open(path).convert("RGB")
        # Apply transforms (resize, augmentation, tensor)
        if self.tf:
            img = self.tf(img)

        # Return:
        #   - image tensor
        #   - label index
        #   - calories as float
        return img, label_to_idx[label], float(cal)



In [ ]:
# ============================================================
# 8. Transforms + DataLoaders
# ============================================================
# Data augmentation & normalization can be extended if needed.
train_tf = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
])

val_tf = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
])

train_ds = FoodDataset("dataset/train","train_calories.csv",train_tf)
val_ds   = FoodDataset("dataset/val","val_calories.csv",val_tf)

# DataLoader creates mini-batches to feed into the network.
train_dl = DataLoader(train_ds, batch_size=32, shuffle=True)
val_dl   = DataLoader(val_ds, batch_size=32, shuffle=False)



In [ ]:
# ============================================================
# 9. EfficientNet-B0 (Classification + Regression)
# ============================================================
class FoodNet(nn.Module):
    """
    Model with:
      - EfficientNet-B0 backbone for feature extraction.
      - One head for classification (10 food classes).
      - One head for regression (calorie estimate).
    """
    def __init__(self, num_classes=10):
        super().__init__()
        # create_model with num_classes=0 means we get a backbone
        # that outputs feature vectors instead of final logits.
        self.backbone = timm.create_model("efficientnet_b0", pretrained=True, num_classes=0)
        f = self.backbone.num_features  # feature dimension
        self.cls = nn.Linear(f, num_classes)  # classification head
        self.reg = nn.Linear(f, 1)           # regression head

    def forward(self, x):
        feat = self.backbone(x)         # (batch_size, f)
        logits = self.cls(feat)         # (batch_size, num_classes)
        calories = self.reg(feat)       # (batch_size, 1)
        # calories.squeeze(1) -> (batch_size,)
        return logits, calories.squeeze(1)

model = FoodNet().to(device)

# Optimizer
opt = torch.optim.Adam(model.parameters(), lr=3e-4)

# Loss functions:
#   - CrossEntropyLoss for classification
#   - SmoothL1Loss (Huber) for calorie regression
loss_cls = nn.CrossEntropyLoss()
loss_reg = nn.SmoothL1Loss()



In [ ]:
# ============================================================
# 10. Training + Validation Loops
# ============================================================
def train_epoch():
    """
    One training pass over the entire training dataset.
    Returns average training loss.
    """
    model.train()
    total_loss = 0.0

    for img, lbl, cal in train_dl:
        img, lbl, cal = img.to(device), lbl.to(device), cal.to(device)

        # Reset gradients
        opt.zero_grad()

        # Forward pass: get classification logits + calorie prediction
        o_cls, o_cal = model(img)

        # Total loss = classification loss + scaled regression loss
        loss = loss_cls(o_cls, lbl) + 0.4 * loss_reg(o_cal, cal)

        # Backpropagation
        loss.backward()

        # One step of optimization
        opt.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_dl)
    return avg_loss


def val_epoch():
    """
    One validation pass over the entire validation dataset.
    Returns:
      - average validation loss
      - average classification accuracy across mini-batches
    """
    model.eval()
    total_loss = 0.0
    total_acc = 0.0

    with torch.no_grad():
        for img, lbl, cal in val_dl:
            img, lbl, cal = img.to(device), lbl.to(device), cal.to(device)

            # Forward pass
            o_cls, o_cal = model(img)

            # Compute loss
            loss = loss_cls(o_cls, lbl) + 0.4 * loss_reg(o_cal, cal)
            total_loss += loss.item()

            # Compute batch accuracy
            preds = o_cls.argmax(1)  # predicted labels
            batch_acc = (preds == lbl).float().mean().item()
            total_acc += batch_acc

    avg_loss = total_loss / len(val_dl)
    avg_acc = total_acc / len(val_dl)
    return avg_loss, avg_acc

In [ ]:
# ============================================================
# 11. Train Model + Save Best Checkpoint to Drive
# ============================================================
EPOCHS = 8

best_val_acc = 0.0
best_epoch = -1

print("\n================= TRAINING START =================\n")
for e in range(EPOCHS):
    tr_loss = train_epoch()
    val_loss, val_acc = val_epoch()

    print(f"Epoch {e+1:02d}/{EPOCHS}: "
          f"Train Loss = {tr_loss:.4f} | "
          f"Val Loss = {val_loss:.4f} | "
          f"Val Acc = {val_acc:.4f}")

    # If this epoch gives the best validation accuracy so far,
    # we save the model weights to Google Drive.
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_epoch = e + 1
        torch.save(model.state_dict(), BEST_MODEL_PATH)
        print(f"  --> New best model saved at epoch {best_epoch} "
              f"with Val Acc = {best_val_acc:.4f}")
        print(f"      Path: {BEST_MODEL_PATH}")

print("\n================= TRAINING END =================\n")
print(f"Best Val Acc = {best_val_acc:.4f} at epoch {best_epoch}")

# Save the final model (after all epochs) as well.
torch.save(model.state_dict(), FINAL_MODEL_PATH)
print(f"\nFinal model saved at: {FINAL_MODEL_PATH}\n")


In [ ]:
# ============================================================
# 12. Detailed Metrics (Per-Class & Global) — FIXED VERSION
# ============================================================
def evaluate_metrics():
    model.eval()

    # Per-class accuracy tracking
    correct = {cls: 0 for cls in SELECTED_CLASSES}
    total   = {cls: 0 for cls in SELECTED_CLASSES}

    # Global classification metrics
    all_true_cls = []
    all_pred_cls = []

    # Regression metrics (calories)
    all_true_cal = []
    all_pred_cal = []

    with torch.no_grad():
        for img, lbl, cal in val_dl:
            img = img.to(device)
            lbl = lbl.to(device)
            cal = cal.to(device)

            out_cls, out_cal = model(img)
            preds = out_cls.argmax(1)

            # Store for global metrics
            all_true_cls.extend(lbl.cpu().numpy())
            all_pred_cls.extend(preds.cpu().numpy())

            all_true_cal.extend(cal.cpu().numpy())
            all_pred_cal.extend(out_cal.cpu().numpy())

            # Per-class stats
            for i in range(len(lbl)):
                t = int(lbl[i].item())
                p = int(preds[i].item())
                cls_name = idx_to_label[t]

                if p == t:
                    correct[cls_name] += 1
                total[cls_name] += 1

    # ---------- CLASSIFICATION METRICS ----------
    y_true = np.array(all_true_cls)
    y_pred = np.array(all_pred_cls)

    overall_acc = accuracy_score(y_true, y_pred) * 100.0
    macro_precision = precision_score(y_true, y_pred, average='macro', zero_division=0) * 100.0
    macro_recall    = recall_score(y_true, y_pred, average='macro', zero_division=0) * 100.0
    macro_f1        = f1_score(y_true, y_pred, average='macro', zero_division=0) * 100.0

    print("\n================ CLASS-WISE ACCURACY ================\n")
    for cls in SELECTED_CLASSES:
        if total[cls] == 0:
            print(f"{cls:15s}: No validation samples")
        else:
            acc = 100.0 * correct[cls] / total[cls]
            print(f"{cls:15s}: {acc:.2f}% ({correct[cls]}/{total[cls]})")

    print("\n================ GLOBAL CLASSIFICATION METRICS ================\n")
    print(f"Overall Accuracy     : {overall_acc:.2f}%")
    print(f"Macro Precision      : {macro_precision:.2f}%")
    print(f"Macro Recall         : {macro_recall:.2f}%")
    print(f"Macro F1-score       : {macro_f1:.2f}%\n")

    # ---------- SAFE CLASSIFICATION REPORT ----------
    print("Classification Report (per class):\n")

    print(classification_report(
        y_true,
        y_pred,
        labels=list(range(len(SELECTED_CLASSES))),  # ensures 10 classes always expected
        target_names=SELECTED_CLASSES,
        zero_division=0
    ))

    # ---------- CONFUSION MATRIX (FIXED WITH PADDING) ----------
    cm = confusion_matrix(y_true, y_pred, labels=list(range(len(SELECTED_CLASSES))))

    # If confusion_matrix returns smaller shape (e.g., 9x9), pad to 10x10
    expected = len(SELECTED_CLASSES)
    if cm.shape != (expected, expected):
        cm_full = np.zeros((expected, expected), dtype=int)
        cm_full[:cm.shape[0], :cm.shape[1]] = cm
        cm = cm_full

    cm_df = pd.DataFrame(cm, index=SELECTED_CLASSES, columns=SELECTED_CLASSES)

    print("\nConfusion Matrix (rows=true, cols=predicted):\n")
    print(cm_df)

    # ---------- REGRESSION METRICS ----------
    y_true_cal = np.array(all_true_cal)
    y_pred_cal = np.array(all_pred_cal)

    mae  = mean_absolute_error(y_true_cal, y_pred_cal)
    rmse = np.sqrt(mean_squared_error(y_true_cal, y_pred_cal))
    r2   = r2_score(y_true_cal, y_pred_cal)

    print("\n================ CALORIE REGRESSION METRICS ================\n")
    print(f"MAE  (Mean Abs Error) : {mae:.4f}")
    print(f"RMSE (Root MSE)       : {rmse:.4f}")
    print(f"R^2 Score             : {r2:.4f}")
    print("\n=============================================================\n")

# Run metrics evaluation
evaluate_metrics()


In [ ]:
# ============================================================
# 13. Random Predictions Gallery
# ============================================================
def show_random_predictions(n=8):
    """
    Shows random validation images with:
      - True class
      - Predicted class
      - True calories
      - Predicted calories
    """
    model.eval()
    plt.figure(figsize=(16, 8))

    # Sample n random indices from the validation dataset
    idxs = random.sample(range(len(val_ds)), n)

    for i, ds_idx in enumerate(idxs):
        img, lbl, cal = val_ds[ds_idx]
        inp = img.unsqueeze(0).to(device)

        with torch.no_grad():
            o_cls, o_cal = model(inp)

        pred_label_idx = o_cls.argmax(1).item()
        pred_label_name = idx_to_label[pred_label_idx]

        np_img = img.permute(1,2,0).numpy()

        plt.subplot(2, n//2, i+1)
        plt.imshow(np_img)
        plt.axis("off")
        plt.title(
            f"True: {idx_to_label[lbl]}\n"
            f"Pred: {pred_label_name}\n"
            f"T Cal: {cal:.0f} | P Cal: {float(o_cal):.0f}"
        )

    plt.tight_layout()
    plt.show()

show_random_predictions()


In [ ]:
# ============================================================
# 14. Grad-CAM (for visual explanation)
# ============================================================
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget

class ClassifierWrapper(nn.Module):
    """
    Wraps the model so that Grad-CAM only sees the classification output.
    This is necessary because our model has two outputs (class + calories).
    """
    def __init__(self, m):
        super().__init__()
        self.m = m

    def forward(self, x):
        o_cls, _ = self.m(x)
        return o_cls

wrapped = ClassifierWrapper(model)

# We choose a target layer from the backbone (last conv layer) for Grad-CAM
target_layers = [model.backbone.conv_head]
cam = GradCAM(model=wrapped, target_layers=target_layers)

def show_gradcam(idx=0):
    """
    Shows the original image and the Grad-CAM heatmap overlay for
    a given index in the validation dataset.
    """
    model.eval()
    img, lbl, cal = val_ds[idx]
    inp = img.unsqueeze(0).to(device)

    # Convert tensor image to float32 numpy [0,1] for visualization
    rgb = img.permute(1,2,0).numpy().astype(np.float32)

    # Target is: "focus Grad-CAM on the true class index"
    targets = [ClassifierOutputTarget(lbl)]

    grayscale_cam = cam(input_tensor=inp, targets=targets)[0]
    cam_img = show_cam_on_image(rgb, grayscale_cam, use_rgb=True)

    plt.figure(figsize=(8,4))
    plt.subplot(1,2,1)
    plt.imshow(rgb)
    plt.title(f"Original ({idx_to_label[lbl]})")
    plt.axis("off")

    plt.subplot(1,2,2)
    plt.imshow(cam_img)
    plt.title("Grad-CAM")
    plt.axis("off")
    plt.tight_layout()
    plt.show()

# Example Grad-CAM visualization
show_gradcam(0)




In [ ]:
# ============================================================
# 15. Upload an Image → Predict
# ============================================================
def predict_uploaded():
    """
    Allows you to upload an image from your local machine,
    then:
      - Runs it through the model,
      - Prints predicted class + calories,
      - Displays the image.
    """
    uploaded = files.upload()
    for fn in uploaded.keys():
        img = Image.open(fn).convert("RGB")
        t = val_tf(img).unsqueeze(0).to(device)

        with torch.no_grad():
            o_cls, o_cal = model(t)

        pred_idx = o_cls.argmax(1).item()
        pred_label = idx_to_label[pred_idx]

        print("\nUploaded file:", fn)
        print("Predicted Class   :", pred_label)
        print("Predicted Calories:", float(o_cal.item()))

        plt.imshow(img)
        plt.axis("off")
        plt.show()

# To use this in Colab, uncomment and run:
# predict_uploaded()